In [1]:
!pip  install torch transformers pytorch_lightning fugashi ipadic unidic_lite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 48.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 43.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.3/695.3 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.1 MB/s eta 0:00:00
  Created wheel for ipadic: filename=ipadic-1.0.0-py3-none-any.whl size=13556705 sha256=8e58495434ed2ac0d446579a7a792bc1ff8b56ffc039b80f82c7d1fad876294e
  Stored in directory: /root/.cache/pip/wheels/44/56/37/f543963822b85260c9f948df8fac8c20169c80dc71b24dc407
  Created wheel for unidic_lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658818 sha256=584e29552f5104c9a7cc50d6792edbf15d4abeded5558a26061b174706b7d078
  Stored in directory: /root/.cache/pip/wheels/b7/fd/e9/ea4459b868e6d2

In [2]:
!pip install pytorch_lightning  torchmetrics

In [3]:
import torch, transformers
print(torch.__version__)
print(transformers.__version__)

2.1.1+cu121
4.35.2


In [4]:
from transformers import AutoTokenizer, AutoModel
import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import unicodedata
import torch.nn as nn
import numpy as np
import random
import torchmetrics.functional as FM
from torchmetrics.functional import precision, recall, f1_score
from torchmetrics.classification import BinaryF1Score

model_path = "line-corporation/line-distilbert-base-japanese"

# tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)


def set_seed(seed=42):
    random.seed(seed)                          # Python標準の乱数
    np.random.seed(seed)                       # Numpyの乱数
    torch.manual_seed(seed)                    # CPU用
    torch.cuda.manual_seed(seed)               # GPU用
    torch.cuda.manual_seed_all(seed)           # マルチGPU用
    torch.backends.cudnn.deterministic = True  # 決定論的アルゴリズムを使用
    torch.backends.cudnn.benchmark = False     # 最適化をオフにして再現性を優先

set_seed(42)
SEED = 42

In [5]:
class GapProjectorLearnable(nn.Module):
    def __init__(self, L=2, R=2, mode="concat"):
        super().__init__()
        
        self.L = L
        self.R = R
        
        self.weight_L = nn.Parameter(torch.zeros(self.L))
        self.weight_R = nn.Parameter(torch.zeros(self.R))
        
        self.mode = mode
    def forward(self, x): # x(B, T, H)bertの出力結果
        B, T, H = x.shape
        device, dtype = x.device, x.dtype
        
        padL = torch.zeros((B, self.L, H), device=device, dtype=dtype)
        padR = torch.zeros((B, self.R, H), device=device, dtype=dtype)
        
        padx = torch.cat([padL, x, padR], dim= 1) #(B, (L+T+R), H)
        
        nor_weight_L = torch.softmax(self.weight_L, dim=0) if self.L > 0 else None
        nor_weight_R = torch.softmax(self.weight_R, dim=0) if self.R > 0 else None
        
        left = torch.zeros((B, T+1, H), device=device, dtype=dtype)
        #左にスライドしていく、縦に計算していくj
        for k in range(1, self.L+1):
            left = left + nor_weight_L[k-1] * padx[:, self.L-k : self.L-k+T+1, :]
            
        right = torch.zeros((B, T+1, H), device=device, dtype=dtype)
        #右にスライドしていく
        for k in range(self.R):
            right = right + nor_weight_R[k] * padx[:, self.L+k : self.L+k+T+1, :]
        
        if self.mode == "concat":
            y = torch.cat([left, right], dim=-1) #(B, T+1, 2*H)
        else:
            y = (left + right) / 2 #(B, T+1, H)
            
        return y
            
        

In [6]:
import unicodedata
import re
import bisect
class Tokenizer(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_path,trust_remote_code=True)
        self.special_tokens_ids = self.tokenizer.all_special_ids
        
        
        SYMBOL = r"[()（）]"
        self.pat = re.compile(SYMBOL)
        
        self.symbol_dict = {"(":1, ")":2}
        
    def _strip_parens_with_prefix_count(self, text: str):
        """
        括弧だけを除去して text_no を作る。
        ついでに i 文字目までに除去された括弧の累積数と、元の括弧の(位置,文字)を返す。
        hits: [(5, '('), (10, ')')]何文字目に丸括弧があるかどうか
        """
        
        
        
        hits = [(m.start(), m.group(0)) for m in self.pat.finditer(text)] #各記号の種類と、位置を求める

        
            
        no_symbol_text = self.pat.sub("", text)#文章から記号を抜く
        
        if not isinstance(no_symbol_text, list):
            no_symbol_text = [no_symbol_text]
        
        return hits, no_symbol_text
        
    
    def _make_prefix(self, text):
        """
        丸括弧があるとその次から+1をしていく？
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2]

        """
        
        remove_prefix = [0] * len(text)
        
        
        prefix_counter = 0
        for i, t in enumerate(text):
            remove_prefix[i] = prefix_counter
            if self.pat.fullmatch(t):
                prefix_counter += 1
                
        
        return remove_prefix
        
    def make_label_and_encode(self, text, max_length=128):

        text = self.normalize_text(text)

        # ① 括弧ありで tokenize（教師用）
        encode_A = self.tokenizer(
            text,
            max_length=max_length,
            add_special_tokens=True, # [CLS]や[SEP]をつける
            return_tensors="pt"
        )

        input_ids_A = encode_A["input_ids"][0]
        tokens_A = self.tokenizer.convert_ids_to_tokens(input_ids_A)
        

        # ② 括弧削除トークン列B
        tokens_B = []
        for tok in tokens_A:
            if any(p in tok for p in ["(", ")", "（", "）"]):
                continue
            tokens_B.append(tok)

        T = len(tokens_B)
        gap_labels = [[0.0, 0.0] for _ in range(T + 1)]

        # ③ Aをもう一度走査して gap を決定
        b_index = 0
        for tok in tokens_A:
            if "(" in tok or "（" in tok:
                gap_labels[b_index][0] = 1.0
            elif ")" in tok or "）" in tok:
                gap_labels[b_index][1] = 1.0
            else:
                b_index += 1

        # ④ BをIDに戻す
        input_ids_B = self.tokenizer.convert_tokens_to_ids(tokens_B)

        # padding
        pad_id = self.tokenizer.pad_token_id
        if len(input_ids_B) < max_length:
            input_ids_B += [pad_id] * (max_length - len(input_ids_B))
        else:
            input_ids_B = input_ids_B[:max_length]

        # gapも同様にpadding
        target_len = max_length + 1
        if len(gap_labels) < target_len:
            gap_labels += [[0.0,0.0]] * (target_len - len(gap_labels))
        else:
            gap_labels = gap_labels[:target_len]

        return {
            "input_ids": torch.tensor(input_ids_B),
            "attention_mask": torch.tensor(
                [1 if i != pad_id else 0 for i in input_ids_B]
            ),
            "labels": torch.tensor(gap_labels, dtype=torch.float)
        }

    def make_encode(self, text, max_length = None, return_tensors=None):
        
        encode = self.tokenizer(text,
                                max_length=max_length,
                                padding="max_length" if max_length else "do_not_pad",
                                truncation=True if max_length else None)
        
        if return_tensors == "pt":
            encode = {k:torch.tensor([v]) for k, v in encode.items()}
            
        return encode
        
    def normalize_text(self, text):
        return unicodedata.normalize("NFKC", text)
        

In [15]:
tok = Tokenizer(model_path)

In [16]:
class NewDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.no_paren_data = [d for d in data if d["label"] == 0]
        self.paren_data = [d for d in data if d["label"] == 1]
        self.tokenizer = tokenizer
    
    def __getitem__(self,idx):
        
        if idx % 2 == 0:
            d = self.no_paren_data[idx % len(self.no_paren_data)]
        else:
            d = self.paren_data[idx % len(self.paren_data)]
        
        encode = self.tokenizer.make_label_and_encode(d["text"])
        
        return encode
    
    def __len__(self):
        return len(self.data)

In [13]:
import json
from sklearn.model_selection import train_test_split
import random
from collections import Counter

data = []
with open("sentences.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))
    
data_no = [d for d in data if d["label"] == 0]
data_pa = [d for d in data if d["label"] == 1]

print(*data_pa[:100], sep='\n')


{'text': '冷蔵保存の場合はなるべく早めに(5日以内)、冷凍保存は1ヶ月以内ににお召し上がりください。', 'label': 1}
{'text': '(3)ニンニクをつぶし、鍋にオリーブオイル(大さじ3)をいれにんにくをいためて香りを出してから、たまねぎを10分程度いためます。', 'label': 1}
{'text': '鍋(実際は、適当な鍋がなかったのでフライパン)に入れ、水を入れ、沸騰させました。', 'label': 1}
{'text': '皮をむく(1)。', 'label': 1}
{'text': '皮をむく(2)。', 'label': 1}
{'text': 'みじん切り(1)。', 'label': 1}
{'text': 'みじん切り(2)。', 'label': 1}
{'text': 'くし形切り(1)。', 'label': 1}
{'text': 'くし形切り(2)。', 'label': 1}
{'text': '縦薄切りにする場合、縦半分に切って、切り口を下にして縦に置き、繊維に沿って、端から料理に応じた厚さに切ります(画像のもの)。', 'label': 1}
{'text': '(4)火を止め、下茹でした葉と塩を入れてご飯を混ぜ、蓋をして10分蒸らします。', 'label': 1}
{'text': '今度は温経湯(ウンケイトウ)といって、やはり血の流れを改善する薬です。', 'label': 1}
{'text': '呉茱萸(ゴシュユ)という、先生曰く、くさーいのが入っているんですが、私には全然くさく感じないんです。', 'label': 1}
{'text': '鍋に(1)の大根を入れ、かぶるくらいの水を加えて火にかける。', 'label': 1}
{'text': '(2)鍋に大根を入れて浸るくらいのお米の研ぎ汁を用意して下茹でする。', 'label': 1}
{'text': '1、鍋に湯を沸かし、少量の塩(分量外)を加えてその中に鶏肉を入れて弱火でゆっくり茹で火が通ったら取り出し、再び火を強め沸騰したらもやしを20秒加えてざるにあける。', 'label': 1}
{'text': 'そうだ、9月にフェリア(祭り)があるからおいで、おいで。', 'label': 1}
{'text': 'ノリオは焦

In [6]:
import re
import json
import random
from typing import List, Dict, Any, Optional
from tqdm.notebook import tqdm


# =========================
# 1. 基本設定
# =========================

EMOTION_WORDS = {
    "笑", "爆", "泣", "汗", "怒", "拍手", "苦笑", "失礼", "照", "驚"
}

UNIT_WORDS = {
    "g", "kg", "mg", "ml", "l", "L", "cm", "mm", "km",
    "秒", "分", "時間", "日", "週間", "ヶ月", "か月", "月", "年",
    "回", "人", "個", "枚", "本", "杯", "大さじ", "小さじ",
    "度", "円", "キロ", "リットル", "トン", "a"
}

CONDITION_WORDS = {
    "以内", "以外", "程度", "くらい", "ぐらい", "約", "分量外",
    "画像のもの", "品質による", "場合", "不都合", "昭和", "平成", "神亀元年"
}

SUPPLEMENT_CUES = {
    "実際は", "または", "という", "いわゆる", "先生曰く", "のでは無く", "では無く",
    "なかったので", "もとは", "写真", "祭り", "超高圧"
}


# =========================
# 2. 丸括弧抽出
# =========================

def extract_parentheses_spans(text: str) -> List[Dict[str, Any]]:
    """
    半角の ( ) を対象に、文中の丸括弧ペアを抽出する。
    """
    spans = []
    stack = []

    for i, ch in enumerate(text):
        if ch == "(":
            stack.append(i)
        elif ch == ")" and stack:
            open_idx = stack.pop()
            content = text[open_idx + 1:i]
            spans.append({
                "open": open_idx,
                "close": i,
                "content": content,
                "surface": text[open_idx:i + 1],
            })

    spans.sort(key=lambda x: x["open"])
    return spans


# =========================
# 3. 判定補助
# =========================

def is_step_number(content: str, open_idx: int, text: str) -> bool:
    c = content.strip()
    if re.fullmatch(r"\d+", c):
        if open_idx <= 3:
            return True
        before = text[max(0, open_idx - 2):open_idx]
        after = text[open_idx:min(len(text), open_idx + 6)]
        if "の" in after or before.endswith("、") or before.endswith(" "):
            return True
    return False


def is_emotion_comment(content: str) -> bool:
    c = content.strip()
    if c in EMOTION_WORDS:
        return True
    if re.fullmatch(r"[!！?？wWｗ笑泣汗爆]+", c):
        return True
    return False


def is_reading_like(content: str, prev_text: str, next_text: str) -> bool:
    c = content.strip()

    if re.fullmatch(r"[ァ-ヶー・]+", c):
        return True
    if re.fullmatch(r"[ぁ-ゖァ-ヶー・]+", c):
        return True

    if re.search(r"[一-龥々]", prev_text[-3:]) and re.fullmatch(r"[ぁ-ゖァ-ヶー・]{1,12}", c):
        return True

    if len(c) <= 6 and re.fullmatch(r"[一-龥ぁ-ゖァ-ヶー]+", c):
        if re.search(r"[一-龥々]", prev_text[-3:]):
            return True

    return False


def is_unit_or_condition(content: str) -> bool:
    c = content.strip()

    if re.search(r"\d", c):
        for w in UNIT_WORDS | CONDITION_WORDS:
            if w in c:
                return True

    for w in CONDITION_WORDS:
        if w in c:
            return True

    for w in UNIT_WORDS:
        if w in c and re.search(r"\d", c):
            return True

    if "どちらでも" in c or "による" in c or "画像" in c:
        return True

    return False


def is_supplement_explanation(content: str) -> bool:
    c = content.strip()

    if len(c) >= 5 and any(cue in c for cue in SUPPLEMENT_CUES):
        return True

    if "、" in c or "。" in c:
        return True

    if re.search(r"(は|が|を|に|で|と|も|より|など)", c) and len(c) >= 6:
        return True

    return False


# =========================
# 4. タグ付け
# =========================

def tag_parenthesis(text: str, span: Dict[str, Any]) -> str:
    open_idx = span["open"]
    close_idx = span["close"]
    content = span["content"]

    prev_text = text[:open_idx]
    next_text = text[close_idx + 1:]

    if is_step_number(content, open_idx, text):
        return "step"

    if is_emotion_comment(content):
        return "emotion"

    if is_reading_like(content, prev_text, next_text):
        return "reading"

    if is_unit_or_condition(content):
        return "unit_condition"

    if is_supplement_explanation(content):
        return "supplement"

    return "other"


def annotate_record(record: Dict[str, Any]) -> Dict[str, Any]:
    text = record.get("text", "")
    spans = extract_parentheses_spans(text)

    annotated_spans = []
    for sp in spans:
        annotated_spans.append({
            **sp,
            "tag": tag_parenthesis(text, sp)
        })

    out = dict(record)
    out["paren_spans"] = annotated_spans

    tags = sorted(set(sp["tag"] for sp in annotated_spans))
    if len(tags) == 0:
        out["sentence_tag"] = "no_parentheses"
    elif len(tags) == 1:
        out["sentence_tag"] = tags[0]
    else:
        out["sentence_tag"] = "mixed"

    return out


# =========================
# 5. jsonl 読み込み
# =========================

def count_jsonl_lines(path: str) -> int:
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def load_jsonl(
    path: str,
    max_records: Optional[int] = None,
    shuffle: bool = False,
    seed: int = 42,
    show_progress: bool = True,
) -> List[Dict[str, Any]]:
    """
    jsonl を読み込む。
    max_records:
        - None のとき全件
        - int のときその件数まで使う
    shuffle:
        - True なら読み込み後にシャッフル
    """
    total_lines = count_jsonl_lines(path)
    records = []

    with open(path, "r", encoding="utf-8") as f:
        iterator = f
        if show_progress:
            iterator = tqdm(f, total=total_lines, desc="Loading jsonl")

        for line_no, line in enumerate(iterator, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if isinstance(rec, dict):
                    records.append(rec)
            except Exception as e:
                print(f"[WARN] line {line_no} を読み込めませんでした: {e}")

    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(records)

    if max_records is not None:
        records = records[:max_records]

    return records


# =========================
# 6. jsonl 保存
# =========================

def save_jsonl(records: List[Dict[str, Any]], path: str, show_progress: bool = True) -> None:
    iterator = records
    if show_progress:
        iterator = tqdm(records, total=len(records), desc="Saving jsonl")

    with open(path, "w", encoding="utf-8") as f:
        for rec in iterator:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


# =========================
# 7. 集計
# =========================

def summarize_tags(records: List[Dict[str, Any]]) -> None:
    from collections import Counter

    span_counter = Counter()
    sentence_counter = Counter()

    for rec in records:
        sentence_counter[rec.get("sentence_tag", "unknown")] += 1
        for sp in rec.get("paren_spans", []):
            span_counter[sp["tag"]] += 1

    print("=== sentence_tag counts ===")
    for k, v in sentence_counter.most_common():
        print(f"{k}: {v}")

    print("\n=== paren tag counts ===")
    for k, v in span_counter.most_common():
        print(f"{k}: {v}")


# =========================
# 8. メイン処理
# =========================

def run_auto_tagging(
    input_path: str,
    output_path: str,
    max_records: Optional[int] = None,
    shuffle: bool = False,
    seed: int = 42,
    show_progress: bool = True,
) -> List[Dict[str, Any]]:
    """
    全体実行関数

    Parameters
    ----------
    input_path : str
        入力 jsonl ファイル
    output_path : str
        出力 jsonl ファイル
    max_records : Optional[int]
        処理する件数
        None なら全件
    shuffle : bool
        True なら読み込み後にシャッフル
    seed : int
        シャッフル時の乱数シード
    show_progress : bool
        tqdm を表示するか
    """
    total_lines = count_jsonl_lines(input_path)
    print(f"入力ファイル総件数: {total_lines:,}")

    if max_records is None:
        print("処理件数: 全件")
    else:
        print(f"処理件数: {min(max_records, total_lines):,}")

    print(f"シャッフル: {shuffle}")
    if shuffle:
        print(f"seed: {seed}")

    records = load_jsonl(
        path=input_path,
        max_records=max_records,
        shuffle=shuffle,
        seed=seed,
        show_progress=show_progress,
    )

    iterator = records
    if show_progress:
        iterator = tqdm(records, total=len(records), desc="Tagging records")

    tagged_records = [annotate_record(rec) for rec in iterator]

    summarize_tags(tagged_records)
    save_jsonl(tagged_records, output_path, show_progress=show_progress)

    print(f"\n保存完了: {output_path}")
    return tagged_records


In [7]:
tagged_records = run_auto_tagging(
    input_path="sentences.jsonl",
    output_path="tagged_sample_1000.jsonl",
    max_records=1000,
    shuffle=True,
    seed=42,
)


入力ファイル総件数: 2,000,142
処理件数: 1,000
シャッフル: True
seed: 42


Loading jsonl:   0%|          | 0/2000142 [00:00<?, ?it/s]

Tagging records:   0%|          | 0/1000 [00:00<?, ?it/s]

=== sentence_tag counts ===
no_parentheses: 503
reading: 150
supplement: 119
other: 115
emotion: 54
unit_condition: 28
mixed: 21
step: 10

=== paren tag counts ===
reading: 188
other: 154
supplement: 132
emotion: 55
unit_condition: 30
step: 11


Saving jsonl:   0%|          | 0/1000 [00:00<?, ?it/s]


保存完了: tagged_sample_1000.jsonl


In [18]:
import json
from sklearn.model_selection import train_test_split
import random
from collections import Counter

USE_SENTENCE_NUM = 100000
data = []
with open("sentences.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))
    
data_no = [d for d in data if d["label"] == 0][:USE_SENTENCE_NUM]
data_pa = [d for d in data if d["label"] == 1][:USE_SENTENCE_NUM]

data = data_no + data_pa

shuffled_data = random.sample(data, k=len(data))

labels_ratio = [d["label"] for d in shuffled_data]

train_val_data, test_data = train_test_split(
            shuffled_data,
            test_size = 0.2,
            random_state = SEED,
            stratify = labels_ratio)

train_val_labels_ratio = [d["label"] for d in train_val_data]

train_data, val_data = train_test_split(
            train_val_data,
            test_size = 0.2,
            random_state = SEED,
            stratify=train_val_labels_ratio)

train_labels = [d["label"] for d in train_data] 
val_labels = [d["label"] for d in val_data] 
test_labels = [d["label"] for d in test_data] 

print(f"trainデータサイズ {len(train_labels)}")
print(f"validデータサイズ{len(val_labels)}")
print(f"testデータサイズ{len(test_labels)}")
train = Counter(train_labels)
val = Counter(val_labels)
test = Counter(test_labels)

print(f"学習データの比率 | 括弧なし {train[0] / (train[0] + train[1])} | 括弧あり {train[1] / (train[0] + train[1])}")
print(f"検証データの比率 | 括弧なし {val[0] / (val[0] + val[1])} | 括弧あり {val[1] / (val[0] + val[1])}")
print(f"テストデータの比率 | 括弧なし {test[0] / (test[0] + test[1])} | 括弧あり {test[1] / (test[0] + test[1])}")

trainデータサイズ 128000
validデータサイズ32000
testデータサイズ40000
学習データの比率 | 括弧なし 0.5 | 括弧あり 0.5
検証データの比率 | 括弧なし 0.5 | 括弧あり 0.5
テストデータの比率 | 括弧なし 0.5 | 括弧あり 0.5


In [19]:
train_dataset = NewDataset(train_data, tok)
val_dataset = NewDataset(val_data, tok)
test_dataset = NewDataset(test_data, tok)

In [20]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=256)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=256)

In [21]:
class GapClassification(pl.LightningModule):
    
    def __init__(self, model_path, tok,L=5, R=5, lr=1e-5, mode="concat"):
        super().__init__()
        self.save_hyperparameters()
        
        self.model = AutoModel.from_pretrained(model_path)
        
                # 🔥 BERT凍結
#         for param in self.model.parameters():
#             param.requires_grad = False
            
        self.gap_pro = GapProjectorLearnable(L=L, R=R, mode=mode)
        
        self.val_open_f1 = BinaryF1Score()
        self.val_close_f1 = BinaryF1Score()
        
        if mode == "concat":
            input_size = self.model.config.hidden_size * 2
        else:
            input_size = self.model.config.hidden_size
            
        self.head_open = nn.Sequential(
                nn.Dropout(0.1),
                nn.Linear(input_size, 1)
        )
        
        self.head_close = nn.Sequential(
                nn.Dropout(0.1),
                nn.Linear(input_size+1, 1)
        )
        
        self.loss_fct = nn.BCEWithLogitsLoss(reduction="none")
        self.tokenizer = tok.tokenizer
        
    def forward(self, batch):
        
        output = self.model(**batch)
        scores = output.last_hidden_state # (B, T, H)
        
        gap_feats = self.gap_pro(scores)
        
        open_logits = self.head_open(gap_feats)
        #1つのスコアの値
        open_probs = torch.sigmoid(open_logits)
        
        # open_state = torch.cumsum(open_probs, dim=1)
        
        close_input = torch.cat([gap_feats, open_probs], dim=-1)
        
        close_logits = self.head_close(close_input)
        
        logits = torch.cat([open_logits, close_logits], dim=-1)
        
        return logits
        
        
    def training_step(self, batch , batch_idx):
        
        labels = batch.pop("labels")  # (B, T+1, 2)
        attention_mask = batch["attention_mask"]  # (B, T)
        
        logits = self(batch)
         
    
        # 🔥 gap用mask作成
        gap_mask = attention_mask.clone()
        gap_mask[:, 0] = 0
        gap_mask = torch.cat(
            [gap_mask, torch.zeros_like(gap_mask[:, :1])],
            dim=1
        )
        
        gap_mask = gap_mask.unsqueeze(-1)  # (B, T+1, 1)

        loss_raw = self.loss_fct(logits, labels.float())
        loss = (loss_raw * gap_mask).sum() / (gap_mask.sum()*2)

        self.log("train_loss", loss)
        return loss
        
        
        
        
    from torchmetrics.functional import precision, recall, f1_score

    # =========================
    # VALIDATION
    # =========================
    def validation_step(self, batch, batch_idx):

        labels = batch.pop("labels")
        attention_mask = batch["attention_mask"]

        logits = self(batch)
        probs = torch.sigmoid(logits)
        
        #閾値を0.5とする
        open_pred = (probs[...,0] > 0.5)
        close_pred = (probs[...,1] > 0.5)
        
        correct_open = (open_pred == labels[...,0].bool())
        correct_close = (close_pred == labels[...,1].bool())
        
        pair_correct = (correct_open & correct_close)


        gap_mask = attention_mask.clone()
        gap_mask[:, 0] = 0  # CLS除外
        gap_mask = torch.cat(
            [gap_mask, torch.zeros_like(gap_mask[:, :1])],
            dim=1
        )
        gap_mask = gap_mask.unsqueeze(-1)
        

        loss_raw = self.loss_fct(logits, labels.float())
        loss = (loss_raw * gap_mask).sum() / (gap_mask.sum()*2)

        self.log("val_loss", loss)


        mask = gap_mask.squeeze(-1) == 1
        
        is_correct_or_ignored = pair_correct | (~mask)
        sequence_perfect = is_correct_or_ignored.all(dim=1).float()
        
        self.val_open_f1.update(probs[...,0][mask], labels[...,0][mask].long())
        self.val_close_f1.update(probs[...,1][mask], labels[...,1][mask].long())

        self.log("val_open_f1", self.val_open_f1, on_step=False, on_epoch=True)
        self.log("val_close_f1", self.val_close_f1, on_step=False, on_epoch=True)
        self.log("val_sequence_em", sequence_perfect.mean(), on_step=False, on_epoch=True)
        return loss
    
    def test_step(self, batch, batch_idx):

        labels = batch.pop("labels")          # (B, T+1, 2)
        attention_mask = batch["attention_mask"]

        logits = self(batch)
        probs = torch.sigmoid(logits)

        gap_mask = attention_mask.clone()
        gap_mask[:, 0] = 0  # CLS除外
        gap_mask = torch.cat(
            [gap_mask, torch.zeros_like(gap_mask[:, :1])],
            dim=1
        )
        gap_mask = gap_mask.unsqueeze(-1)

        # ===== loss =====
        loss_raw = self.loss_fct(logits, labels)
        loss = (loss_raw * gap_mask).sum() / (gap_mask.sum() + 1e-8)
        self.log("test_loss", loss)

        # ===== 0.5 threshold =====
        preds = (probs > 0.5).float()

        mask = gap_mask.squeeze(-1) == 1

        # =========================
        # OPEN
        # =========================
        open_pred = preds[..., 0][mask]
        open_label = labels[..., 0][mask]

        open_f1 = f1_score(open_pred, open_label, task="binary")
        open_precision = precision(open_pred, open_label, task="binary")
        open_recall = recall(open_pred, open_label, task="binary")

        self.log("test_open_f1", open_f1)
        self.log("test_open_precision", open_precision)
        self.log("test_open_recall", open_recall)

        # =========================
        # CLOSE
        # =========================
        close_pred = preds[..., 1][mask]
        close_label = labels[..., 1][mask]

        close_f1 = f1_score(close_pred, close_label, task="binary")
        close_precision = precision(close_pred, close_label, task="binary")
        close_recall = recall(close_pred, close_label, task="binary")

        self.log("test_close_f1", close_f1)
        self.log("test_close_precision", close_precision)
        self.log("test_close_recall", close_recall)

        # =========================
        # False Positive Rate
        # =========================
        non_open_mask = (open_label == 0)
        if non_open_mask.sum() > 0:
            fp_open = open_pred[non_open_mask].float().mean()
        else:
            fp_open = torch.tensor(0.0, device=open_pred.device)
       
        self.log("test_fp_open", fp_open)

        non_close_mask = (close_label == 0)
        if non_close_mask.sum() > 0:
            fp_close = close_pred[non_close_mask].float().mean()
        else:
            fp_close = torch.tensor(0.0, device=close_pred.device)
        self.log("test_fp_close", fp_close)

        return loss

    # =========================
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
    
    # def configure_optimizers(self):
    #     return torch.optim.AdamW(
    #         filter(lambda p: p.requires_grad, self.parameters()),
    #         lr=self.hparams.lr
    #     )

        
# logger_csv = pl.loggers.CSVLogger("outputs", name="Lightning_logs_csv2")
# logger_tb = pl.loggers.TensorBoardLogger("outputs", name="Lightning_logs_tb2")

# check_point = pl.callbacks.ModelCheckpoint(
#                 monitor = "val_loss",
#                 mode = "min",
#                 save_top_k = 1,
#                 save_weights_only = True,
#                 dirpath = "model/"
#                 )

# trainer = pl.Trainer(
#         max_epochs = 5,
#         callbacks = [check_point],
#         accelerator = "gpu",
#         devices = 1,
#         logger = [logger_csv, logger_tb])


# model = GapClassification(model_path = "line-corporation/line-distilbert-base-japanese", tok=tok,L=1, R=1)
# # for name, param in model.named_parameters():
# #     if param.requires_grad:
# #         print(name)
# model.train()
# trainer.fit(model, train_dataloader, val_dataloader)

# result = trainer.test(dataloaders = test_dataloader)
# print(result)

In [24]:
def _resolve_base_tokenizer(tokenizer):
    # Support both project's Tokenizer wrapper and raw HF tokenizer.
    return getattr(tokenizer, "tokenizer", tokenizer)


def make_token(word_list, tokenizer):
    tokenizer = _resolve_base_tokenizer(tokenizer)
    token_list = []
    token_order = []

    # [CLS]トークンを先頭に追加
    token_list.append(tokenizer.cls_token)
    token_order.append(-1)  # CLSトークンはどの単語にも対応しない

    for i, word in enumerate(word_list):
        tokens = tokenizer.tokenize(word)
        token_num = len(tokens)
        word_token_indices = [-10] * token_num
        word_token_indices[0] = i  # 最初のトークンに元の単語インデックスを設定
        # word_token_indices[-1] = i

        token_order.extend(word_token_indices)
        token_list.extend(tokens)

    # [SEP]トークンを末尾に追加
    token_list.append(tokenizer.sep_token)
    token_order.append(-1)  # SEPトークンはどの単語にも対応しない
    

    ids_list = tokenizer.convert_tokens_to_ids(token_list)

    return token_list, token_order, ids_list



    # token_orderから各単語に対応するトークン位置を収集
    word_token_positions = {}
    for token_idx, word_idx in enumerate(token_order):
        if word_idx == -1:
            continue
        word_token_positions.setdefault(word_idx, []).append(token_idx)

    # 音声認識の単語順に、先頭/末尾トークン由来の確率を返す
    word_scores = []
    for word_idx, word in enumerate(words):
        token_indices = word_token_positions.get(word_idx, [])
        if not token_indices:
            continue

        first_token_idx = min(token_indices)
        last_token_idx = max(token_indices)

        # 2次元出力: [open, close]
        open_prob = float(score[first_token_idx, 0].item())
        close_prob = float(score[last_token_idx, 1].item())

        word_scores.append({
            "word": word,
            "probs": [open_prob, close_prob],
            "token_span": [first_token_idx, last_token_idx],
        })

    return word_scores

In [25]:
import torch
import numpy as np

def token_gap_to_word_gap(token_order, gap_scores_2d, token_list, sep_token, pad_token):
    """
    token_order: len=T（CLS/SEP/PAD含む）: 単語先頭トークンだけ>=0、残り-1
    gap_scores_2d: shape (T+1, 2)
    token_list: len=T（idsから復元したトークン列）
    """
    gap_scores_2d = np.asarray(gap_scores_2d)
    T = len(token_order)
    assert gap_scores_2d.shape[0] == T + 1, "gap_scores must be (T+1, 2)"
    assert gap_scores_2d.shape[1] == 2
    assert len(token_list) == T, f"len(token_list)={len(token_list)} != len(token_order)={T}"

    starts = []
    ends = []

    
    i = 0
    while i < T:
        if token_order[i] >= 0:
            start = i
            j = i + 1
            
            # ✅ -1 が続いても SEP/PAD に来たら止める（PAD末尾まで飲み込まない）
            while (
                j < T
                and token_order[j] == -10
                and token_list[j] != sep_token
                and token_list[j] != pad_token
            ):
                j += 1

            end = j - 1
            starts.append(start)
            ends.append(end)
            i = j
        else:
            i += 1

    W = len(starts)
    assert W > 0, "No words found from token_order"

    word_gaps = []
    word_gaps.append(gap_scores_2d[starts[0]])      # 文頭〜word0前

    for k in range(1, W):
        word_gaps.append(gap_scores_2d[starts[k]])  # 単語間

    word_gaps.append(gap_scores_2d[ends[-1] + 1])   # 文末（最後単語の直後）

    return np.asarray(word_gaps)  # (W+1, 2)


def print_gap_scores(word_list, scores_2d, digits=4):
    scores_2d = np.asarray(scores_2d)
    W = len(word_list)
    assert scores_2d.shape == (W + 1, 2), f"scores must be (W+1,2), got {scores_2d.shape}"

    for i in range(W):
        print(f"{scores_2d[i,0]:.{digits}f} {scores_2d[i,1]:.{digits}f}")
        print(word_list[i])

    print(f"{scores_2d[W,0]:.{digits}f} {scores_2d[W,1]:.{digits}f}")


def predict(ids_list, words, token_order, tokenizer, model, digits=4, print_result=True):
    device = next(model.parameters()).device

    input_ids = torch.tensor([ids_list], dtype=torch.long, device=device)

    # ✅ PADを0に
    pad_id = tokenizer.tokenizer.pad_token_id
    attention_mask = (input_ids != pad_id).long()

    assert len(ids_list) == len(token_order), f"len(ids_list)={len(ids_list)} != len(token_order)={len(token_order)}"

    model.eval()
    with torch.no_grad():
        logits = model({"input_ids": input_ids, "attention_mask": attention_mask})  # (1, T+1, 2)
        probs = torch.sigmoid(logits)[0]  # (T+1, 2)

    score = probs.detach().cpu().numpy()
  

    # ✅ ids -> token_list（CLS/SEP/PAD含む）
    token_list = tokenizer.tokenizer.convert_ids_to_tokens(ids_list)
    sep_token = tokenizer.tokenizer.sep_token
    pad_token = tokenizer.tokenizer.pad_token

    word_gap_scores = token_gap_to_word_gap(token_order, score, token_list, sep_token, pad_token)

    word_gap_scores = np.round(word_gap_scores, decimals=digits)
 
    if print_result:
        print_gap_scores(words, word_gap_scores, digits=digits)

    return word_gap_scores

In [32]:
pl_model = GapClassification.load_from_checkpoint("model/epoch=3-step=4800.ckpt")

model = pl_model.cuda()

# text = "私が好きなのは、"
text = ["機械","は","270秒","に","設定","された","動作","を","維持","して","いる。"]
# token = tok.tokenizer.tokenize(text)


token_list, token_order, ids_list = make_token(text, tok.tokenizer)
word_gap_scores = predict(
        ids_list,
        text,
        token_order,
        tok,
        pl_model,
        print_result=False,
    )

print(word_gap_scores)

[[6.20e-03 2.00e-04]
 [2.00e-04 1.00e-04]
 [8.00e-04 1.00e-04]
 [0.00e+00 3.00e-04]
 [4.00e-04 2.00e-04]
 [0.00e+00 1.00e-04]
 [4.00e-04 2.00e-04]
 [0.00e+00 1.00e-04]
 [1.00e-04 8.00e-04]
 [1.00e-04 1.00e-03]
 [1.10e-03 3.50e-03]
 [7.39e-02 1.49e-02]]


In [ ]:
class Tokenizer(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        
        SYMBOL = r"[\(\)]"
        self.pat = re.compile(SYMBOL)
        
        self.symbol_dict = {"(":1, ")":2}
        
    def _strip_parens_with_prefix_count(self, text: str) -> Tuple[str, List[int], List[Tuple[int,str]]]:
        """
        括弧だけを除去して text_no を作る。
        ついでに i 文字目までに除去された括弧の累積数と、元の括弧の(位置,文字)を返す。
        """
        
        
        
        hits = [(m.start(), m.group(0)) for m in self.pat.finditer(text)] #各記号の種類と、位置を求める

        
            
        no_symbol_text = self.pat.sub("", text)#文章から記号を抜く
        
        return hits, no_symbol_text
        
    
    def _make_prefix(self, text):
        
        remove_prefix = [0] * len(text)+1
        
        
        prefix_counter = 0
        for i, t in enumerate(text):
            remove_prefix[i] = prefix_counter
            if self.pat.fullmatch(t):
                prefix_counter += 1
                
        
        return remove_prefix
        
    def make_label_and_encode(self, text, max_length=128):
        text = normalize_text(text)
        
        
        hits,　no_symbol_text = self._strip_parens_with_prefix_count(text)
        
        remove_prefix = self._make_prefix(text)
        
        encode = self.tokenizer(no_symbol_text, max_length=max_length, padding="max_length", truncation=True, return_offsets_mapping=True, return_special_tokens_mask=True)
        
        baseline = [(pos - remove_prefix[pos],  char) for pos, char in hits]#削除した記号の分だけ、ギャップの位置をずらす
           
        
        spmask = encode["special_tokens_mask"][0]
        off = encode["offset_mapping"][0]
        
        start_off = [s for (s, e), sp in zip(off, spmask) if sp == 0]
        
        T = len(start_off) 
        gap_label = [0] * (T + 1)
        
        for pos, char in baseline:
            idx = bisect.bisect_left(start_off, pos)
            idx = min(0, max(idx, T))
            
            new_label = self.symbol_dict(char)
            gap_label = max(gap_label[idx], new_label)
            
        labels = [-100] + gap_labels[:max_length - 2] + [-100]
        labels = labels + [-100] * (max_length - len(labels))
        
        encode["labels"] = labels
        
        return encode
        
    def normalize_text(self, text):
        return unicodedate.normalize("NFKC", text)
        


In [33]:
import re

model_path = "cl-tohoku/bert-base-japanese-whole-word-masking"
tokenizer = AutoTokenizer.from_pretrained(model_path)
text = "私は(いまだね)小学です。"

SYMBOL = r"[\(\)]"
pat = re.compile(SYMBOL)

symbol_dict = {"(":1, ")":2}

# encode = tokenizer(text, return_offsets_mapping=True)
# print(encode["offset_mapping"])
print(tokenizer.is_fast)

False


In [16]:
import numpy as np

arr = np.array([1,2,3])
arr[[1,2]] = 1
print(arr)

[1 1 1]


In [20]:
import re
text = "私は(いまだね)小学です。" 
encode = tok(text, return_offsets_mapping=True, padding="max_length",max_length=54,truncation=True)
print(encode)
print(encode["input_ids"])
print(encode["offset_mapping"])
paren_pat = re.compile(r"[()（）]")
text = paren_pat.sub("", text)
print(text)

{'input_ids': [2, 1127, 3228, 31473, 437, 444, 1015, 3080, 6538, 5129, 3235, 4615, 31911, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'offset_mapping': [(0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 12), (12, 13), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0)

In [ ]:
for pos_base, ch in base_positions:
            gid = bisect.bisect_left(starts, pos_base)   # 0..T
            gid = max(0, min(gid, T))
            
            new_label = self.kind_map.get(ch, 0)
            

In [153]:
tok = Tokenizer(model_path = "nlp-waseda/roberta-base-japanese")

In [154]:
text = "私は(いまだね)小学です。" 

In [155]:
print(tok._strip_parens_with_prefix_count(text))

([(2, '('), (7, ')')], ['私はいまだね小学です。'])


In [156]:
print(tok._make_prefix(text))

[0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]


In [157]:
print(tok.make_label_and_encode(text))

[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
start_off [0, 1, 2, 3, 4, 5, 6, 7, 8, 10]
pos 2 | char (
gap_label 0 | new_label 1
pos 6 | char )
gap_label 0 | new_label 2
{'input_ids': [[2, 1127, 3228, 437, 444, 1015, 3080, 5129, 3235, 4615, 31911, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [147]:
text = "私はいまだね小学です。" 
print(tok.tokenizer.tokenize(text))

['▁私', 'は', 'い', 'ま', 'だ', 'ね', '小', '学', 'です', '。']


In [151]:
text = "蘊"
encode = tok.tokenizer(text, return_special_tokens_mask=True)
print(encode)
print(tok.tokenizer.convert_ids_to_tokens(encode["input_ids"]))

{'input_ids': [2, 275, 1, 3], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1], 'special_tokens_mask': [1, 0, 0, 1]}
['[CLS]', '▁', '[UNK]', '[SEP]']


In [4]:
a = torch.tensor([[1,2,3]])
b = a.squeeze(0)
c = b.squeeze(0)
print(a.size())
print(b.size())
print(c.size())

torch.Size([1, 3])
torch.Size([3])
torch.Size([3])


In [20]:
token_order = [0,1,2,-1,3,-1]
score = np.array([1,2,3,4,5,6])
token_order = np.array(token_order)
mask = token_order != -1
print(mask)
print(score[mask])

[ True  True  True False  True False]
[1 2 3 5]


In [67]:
from collections import Counter
import torch
from tqdm.notebook import tqdm

label_counter = Counter()



for batch in tqdm(train_dataloader):
    labels = batch["labels"]  # (B, T)
    
    # ignore_index (-100) 除外
    mask = labels != -100
    valid_labels = labels[mask]
    
    # CPUに移してflatten
    flat = valid_labels.view(-1).cpu().tolist()
    
    label_counter.update(flat)
    
   

print(label_counter)

  0%|          | 0/10577 [00:00<?, ?it/s]

Counter({0: 42908246, 1: 188059, 2: 187473})


In [55]:
def normalize_text( text):
    return unicodedata.normalize("NFKC", text)

def make_label_and_encode(text, tokenizer,max_length=128):

        text = normalize_text(text)

        # ① 括弧ありで tokenize（教師用）
        encode_A = tokenizer(
            text,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids_A = encode_A["input_ids"][0]
        tokens_A = tokenizer.convert_ids_to_tokens(input_ids_A)

        # ② 括弧トークンを削除して入力用トークン列Bを作る
        tokens_B = []
        gap_labels = []

        b_index = 0

        # gapは B基準なので、最初に0で初期化
        # 長さは「Bのトークン数 + 1」になる
        # まずBを作りながらAを走査する

        # 一時的に B を構築
        for tok in tokens_A:
            if ("(" in tok) or (")" in tok) or ("（" in tok) or ("）" in tok):
                continue
            tokens_B.append(tok)
             
        print(tokens_B)

        T = len(tokens_B)
        gap_labels = [0] * (T + 1)

        # ③ Aをもう一度走査して gap を決定
        b_index = 0
        for tok in tokens_A:
            if "(" in tok or "（" in tok:
                gap_labels[b_index] = 1
            elif ")" in tok or "）" in tok:
                gap_labels[b_index] = 2
            else:
                b_index += 1
        

        # ④ BをIDに戻す
        input_ids_B = tokenizer.convert_tokens_to_ids(tokens_B)

        # padding
        pad_id = tokenizer.pad_token_id
        if len(input_ids_B) < max_length:
            input_ids_B += [pad_id] * (max_length - len(input_ids_B))
        else:
            input_ids_B = input_ids_B[:max_length]

        # gapも同様にpadding
        target_len = max_length + 1
        if len(gap_labels) < target_len:
            gap_labels += [-100] * (target_len - len(gap_labels))
        else:
            gap_labels = gap_labels[:target_len]

        return {
            "input_ids": torch.tensor(input_ids_B),
            "attention_mask": torch.tensor(
                [1 if i != pad_id else 0 for i in input_ids_B]
            ),
            "labels": torch.tensor(gap_labels)
        }

In [13]:
def normalize_text( text):
    return unicodedata.normalize("NFKC", text)

def make_label_and_encode(text, tokenizer,max_length=128):

        text = normalize_text(text)

        # ① 括弧ありで tokenize（教師用）
        encode_A = tokenizer(
            text,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids_A = encode_A["input_ids"][0]
        tokens_A = tokenizer.convert_ids_to_tokens(input_ids_A)
        
#         special_ids = set(tokenizer.all_special_ids)
        
#         # 🔥 special token除外
#         tokens_A = []
#         for tid in input_ids_A:
#             if tid.item() in special_ids:
#                 continue
#             tokens_A.append(
#                 tokenizer.convert_ids_to_tokens(tid.item())
#             )

        # ② 括弧削除トークン列B
        tokens_B = []
        for tok in tokens_A:
            if any(p in tok for p in ["(", ")", "（", "）"]):
                continue
            tokens_B.append(tok)

        T = len(tokens_B)
        gap_labels = [[0.0, 0.0] for _ in range(T + 1)]

        # ③ Aをもう一度走査して gap を決定
        b_index = 0
        for tok in tokens_A:
            if "(" in tok or "（" in tok:
                gap_labels[b_index][0] = 1.0
            elif ")" in tok or "）" in tok:
                gap_labels[b_index][1] = 1.0
            else:
                b_index += 1

        # ④ BをIDに戻す
        input_ids_B = tokenizer.convert_tokens_to_ids(tokens_B)

        # padding
        pad_id = tokenizer.pad_token_id
        if len(input_ids_B) < max_length:
            input_ids_B += [pad_id] * (max_length - len(input_ids_B))
        else:
            input_ids_B = input_ids_B[:max_length]

        # gapも同様にpadding
        target_len = max_length + 1
        if len(gap_labels) < target_len:
            gap_labels += [[0.0,0.0]] * (target_len - len(gap_labels))
        else:
            gap_labels = gap_labels[:target_len]

        return {
            "input_ids": torch.tensor(input_ids_B),
            "attention_mask": torch.tensor(
                [1 if i != pad_id else 0 for i in input_ids_B]
            ),
            "labels": torch.tensor(gap_labels, dtype=torch.float)
        }


In [14]:
tok = AutoTokenizer.from_pretrained(model_path,trust_remote_code=True)
text = "彼は(東京大学)(京都大学)出身である。"
no_text = "彼は東京大学京都大学出身である。"
encode = make_label_and_encode(text, tok,max_length=128)

print(tok.tokenize(no_text))
print(encode["labels"])
print(encode["attention_mask"])

['▁彼', '▁は', '▁東京', '▁大学', '▁京都', '▁大学', '▁出身', '▁で', '▁ある', '▁。']
tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        [1., 0.],
        [0., 0.],
        [1., 1.],
        [0., 0.],
        [0., 1.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.

In [8]:
words = ["私","は","群馬","大学","前橋","市","に","通っ","て","いる"]

encoded = tokenizer(
    words,
    is_split_into_words=True
)

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print(tokens)

['[CLS]', '▁私', '▁は', '▁群馬', '▁大学', '▁前', '橋', '▁市', '▁に', '▁通っ', '▁て', '▁いる', '[SEP]']


In [ ]:
class GapClassification(pl.LightningModule):
    
    def __init__(self, model_path, num_symbol, L=5, R=5, lr=1e-5, mode="concat", ignore_index=-100, weight = None):
        super().__init__()
        self.save_hyperparameters()
        
        self.model = AutoModel.from_pretrained(model_path)
        
        self.num_symbol = num_symbol
        
        self.gap_pro = GapProjectorLearnable(L=L, R=R, mode=mode)
        
        if mode == "concat":
            input_size = self.model.config.hidden_size * 2
        else:
            input_size = self.model.config.hidden_size
            
        self.head = nn.Sequential(
                nn.Dropout(0.2),
                nn.Linear(input_size, 1568),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(1568, num_symbol))
        
        class_weight_tensor = None
        if weight is not None:
            class_weight_tensor = torch.tensor(weight, dtype=torch.float)
        self.loss_fct = nn.CrossEntropyLoss(weight= class_weight_tensor, ignore_index=ignore_index)
        
    def forward(self, batch):
        
        output = self.model(**batch)
        scores = output.last_hidden_state # (B, T, H)
        
        y = self.gap_pro(scores)
        
        y = self.head(y)
        
        return y
        
        
    def training_step(self, batch , batch_idx):
        
        labels = batch.pop("labels")
        
        y = self({key: batch[key] for key in ("input_ids", "attention_mask") if key in batch})
         
    
        ce_loss = self.loss_fct(y.view(-1, self.num_symbol), labels.view(-1))
        
        
        
        loss = ce_loss
        
        self.log("training_loss", loss)
        self.log("training_acc", FM.accuracy(preds, labels, task="multiclass", num_classes = self.num_symbol,ignore_index=-100))
        
        return loss
        
        
        
        
    from torchmetrics.functional import precision, recall, f1_score

    def validation_step(self, batch, batch_idx):
        labels = batch.pop("labels")

        y = self({k: batch[k] for k in ("input_ids", "attention_mask") if k in batch})
        preds = y.argmax(dim=-1)

        loss = self.loss_fct(y.view(-1, self.num_symbol), labels.view(-1))
        self.log("val_loss", loss)

        # ========= ignore_index除去 =========
        mask = labels != -100
        flat_preds = preds[mask]
        flat_labels = labels[mask]

        open_idx = 1
        close_idx = 2

        # ========= 全体 =========
        acc = (flat_preds == flat_labels).float().mean()
        self.log("val_acc", acc)

        macro_f1 = f1_score(
            flat_preds, flat_labels,
            task="multiclass",
            num_classes=self.num_symbol,
            average="macro"
        )
        self.log("val_macro_f1", macro_f1)

        # ========= 開き =========
        self.log("val_open_precision",
                 precision(flat_preds, flat_labels,
                           task="multiclass",
                           num_classes=self.num_symbol,
                           average=None)[open_idx])

        self.log("val_open_recall",
                 recall(flat_preds, flat_labels,
                        task="multiclass",
                        num_classes=self.num_symbol,
                        average=None)[open_idx])

        self.log("val_open_f1",
                 f1_score(flat_preds, flat_labels,
                          task="multiclass",
                          num_classes=self.num_symbol,
                          average=None)[open_idx])

        # ========= 閉じ =========
        self.log("val_close_precision",
                 precision(flat_preds, flat_labels,
                           task="multiclass",
                           num_classes=self.num_symbol,
                           average=None)[close_idx])

        self.log("val_close_recall",
                 recall(flat_preds, flat_labels,
                        task="multiclass",
                        num_classes=self.num_symbol,
                        average=None)[close_idx])

        self.log("val_close_f1",
                 f1_score(flat_preds, flat_labels,
                          task="multiclass",
                          num_classes=self.num_symbol,
                          average=None)[close_idx])

        # ========= False Positive =========
        non_paren_mask = (flat_labels != open_idx) & (flat_labels != close_idx)

        fp_open = (flat_preds[non_paren_mask] == open_idx).float().mean()
        fp_close = (flat_preds[non_paren_mask] == close_idx).float().mean()

        self.log("val_fp_open", fp_open)
        self.log("val_fp_close", fp_close)

        # ========= 構造一致評価 =========
        pair_scores = []
        strict_match_scores = []

        for b in range(preds.size(0)):
            p_seq = preds[b]
            l_seq = labels[b]

            pred_stack = []
            label_stack = []

            correct_pairs = 0
            total_pairs = 0

            for i, (p, l) in enumerate(zip(p_seq, l_seq)):

                if l == -100:
                    continue

                if l == open_idx:
                    label_stack.append(i)
                elif l == close_idx:
                    if label_stack:
                        label_stack.pop()
                        total_pairs += 1

                if p == open_idx:
                    pred_stack.append(i)
                elif p == close_idx:
                    if pred_stack:
                        pred_open = pred_stack.pop()
                        correct_pairs += 1

            if total_pairs > 0:
                pair_scores.append(correct_pairs / total_pairs)

            # 完全一致
            strict = ((p_seq == l_seq) | (l_seq == -100)).float().mean()
            strict_match_scores.append(strict)

        if pair_scores:
            self.log("val_pair_acc", torch.tensor(pair_scores).mean())

        self.log("val_strict_token_acc", torch.stack(strict_match_scores).mean())

        return loss

        
    def test_step(self, batch, batch_idx):
        labels = batch.pop("labels")

        y = self({k: batch[k] for k in ("input_ids", "attention_mask") if k in batch})
        preds = y.argmax(dim=-1)

        loss = self.loss_fct(y.view(-1, self.num_symbol), labels.view(-1))
        self.log("test_loss", loss)

        mask = labels != -100
        flat_preds = preds[mask]
        flat_labels = labels[mask]

        open_idx = 1
        close_idx = 2

        acc = (flat_preds == flat_labels).float().mean()
        self.log("test_acc", acc)

        macro_f1 = f1_score(
            flat_preds, flat_labels,
            task="multiclass",
            num_classes=self.num_symbol,
            average="macro"
        )
        self.log("test_macro_f1", macro_f1)

        # 以下は validation と同様
        self.log("test_open_f1",
                 f1_score(flat_preds, flat_labels,
                          task="multiclass",
                          num_classes=self.num_symbol,
                          average=None)[open_idx])

        self.log("test_close_f1",
                 f1_score(flat_preds, flat_labels,
                          task="multiclass",
                          num_classes=self.num_symbol,
                          average=None)[close_idx])

        non_paren_mask = (flat_labels != open_idx) & (flat_labels != close_idx)
        self.log("test_fp_open",
                 (flat_preds[non_paren_mask] == open_idx).float().mean())

        self.log("test_fp_close",
                 (flat_preds[non_paren_mask] == close_idx).float().mean())

        return loss

        
        
        
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
    
#     def pair_loss(self,preds, labels, open_idx=1, close_idx=2):
#         """開閉ペアの一致度を損失として計算"""
#         batch_size, seq_len = labels.shape
#         loss = 0.0
#         for b in range(batch_size):
#             pred_stack = []
#             label_stack = []
#             correct_pairs = 0
#             total_pairs = 0

#             for i, (p, l) in enumerate(zip(preds[b], labels[b])):
#                 if l == -100:  # ignore_index
#                     continue

#                 if l == open_idx:
#                     label_stack.append(i)
#                 elif l == close_idx:
#                     if label_stack:
#                         label_stack.pop()
#                         total_pairs += 1

#                 if p == open_idx:
#                     pred_stack.append(i)
#                 elif p == close_idx:
#                     if pred_stack:
#                         pred_stack.pop()
#                         correct_pairs += 1

#             if total_pairs > 0:
#                 loss += 1.0 - (correct_pairs / total_pairs)  # 一致率が高いほど loss は小さい

#         return loss / batch_size

        
logger_csv = pl.loggers.CSVLogger("outputs", name="Lightning_logs_csv")
logger_tb = pl.loggers.TensorBoardLogger("outputs", name="Lightning_logs_tb")

check_point = pl.callbacks.ModelCheckpoint(
                monitor = "val_loss",
                mode = "min",
                save_top_k = 1,
                save_weights_only = True,
                dirpath = "model/"
                )

trainer = pl.Trainer(
        max_epochs = 10,
        callbacks = [check_point],
        accelerator = "gpu",
        devices = 1,
        logger = [logger_csv, logger_tb])

weight = [1.0, 10.0, 10.0]
model = GapClassification(model_path = "line-corporation/line-distilbert-base-japanese", num_symbol=3, weight=weight)
model.train()
trainer.fit(model, train_dataloader, val_dataloader)

result = trainer.test(dataloaders = test_dataloader)